In [2]:
import pandas as pd
import os
import re
import time
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed

# CONFIG
RAW_FILE = r"C:\Users\23225632\Documents\Projects\UserLookup\data\Kenya Offrole & CWK Dump_27 FEB.xlsx"
SHEET_NAME = "Sheet1"
HOSTNAME_COL = "Hostname"
OUTPUT_FILE = "output.xlsx"
OUTPUT_SHEET = "user data"

# Thread pool sizing. 20–40 is usually fine for domain calls.
MAX_WORKERS = 32
# Subprocess timeout per call (seconds)
NETUSER_TIMEOUT = 8
# Retries if a call times out or fails
RETRIES = 2
VERBOSE = True  
# END CONFIG

In [ ]:

def log(msg):
    if VERBOSE:
        print(msg)

# -------------------- Load & Normalize --------------------------
df = pd.read_excel(RAW_FILE, sheet_name=SHEET_NAME, engine="openpyxl")

# Split multi-line hostnames into separate rows
df = df.assign(**{
    HOSTNAME_COL: df[HOSTNAME_COL].astype(str).str.split(r'[\n\r]+')
})
df = df.explode(HOSTNAME_COL).reset_index(drop=True)

# Extract AUUID (first sequence of 5+ digits) from Hostname
auuid_series = df[HOSTNAME_COL].astype(str).str.extract(r"(\d{5,})", expand=False)
df["AUUID_digits"] = auuid_series

# Only query valid (non-null) IDs
mask_valid = df["AUUID_digits"].notna()
unique_ids = pd.unique(df.loc[mask_valid, "AUUID_digits"].astype(str))

log(f"Total rows: {len(df)} | Unique AUUIDs to query: {len(unique_ids)}")

# -------------------- Domain Query Function ---------------------
FULLNAME_REGEX = re.compile(r'^\s*Full\s*Name\s*:\s*(.+?)\s*$', re.IGNORECASE)

def extract_staff_name(auuid: str, timeout: int = NETUSER_TIMEOUT):
    """
    Returns display name or None.
    """
    # Ensure auuid is a clean string (no .0, no whitespace)
    auuid = str(auuid).strip()

    # If you run this on Linux/macOS/WSL, 'net' won't exist:
    # Guard to avoid crashing silently.
    if os.name != "nt":
        return None

    cmd = ["net", "user", "/domain", auuid]
    try:
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=timeout,
            shell=False
        )
    except subprocess.TimeoutExpired:
        log(f"[TIMEOUT] {auuid}")
        return None
    except Exception as e:
        log(f"[ERROR] {auuid}: {e}")
        return None

    if result.returncode != 0:
        # Helpful to see stderr if users aren't found
        if result.stderr:
            log(f"[RC{result.returncode}] {auuid} stderr: {result.stderr.strip()}")
        else:
            log(f"[RC{result.returncode}] {auuid} stdout: {result.stdout.strip()[:120]}...")
        return None

    # Try strict regex first
    for line in result.stdout.splitlines():
        m = FULLNAME_REGEX.match(line)
        if m:
            name = m.group(1).strip()
            return name or None

    # Fallback: looser parsing
    for line in result.stdout.splitlines():
        if "Full" in line and "Name" in line and ":" in line:
            try:
                return line.split(":", 1)[1].strip() or None
            except Exception:
                pass

    # Optional extra fallback: look for "Name" field if your domain output differs
    # for line in result.stdout.splitlines():
    #     if line.strip().lower().startswith("name"):
    #         try:
    #             return line.split(":", 1)[1].strip() or None
    #         except Exception:
    #             pass

    return None

def extract_staff_name(auuid):
    command = f'net user /domain "{auuid}"'
    result = subprocess.run(command, capture_output=True, text=True, shell=True)
    for line in result.stdout.splitlines():
        if 'Full Name' in line:
            parts = line.split(':', 1)
            if len(parts) > 1:
                return parts[1].strip()
    return None
# -------------------- Parallel Execution ------------------------
mapping = {}
if len(unique_ids) > 0:
    log(f"Starting parallel lookups with {MAX_WORKERS} workers...")
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(extract_with_retries, uid): uid for uid in unique_ids}
        done_count = 0
        for future in as_completed(futures):
            uid = futures[future]
            try:
                mapping[uid] = future.result()
            except Exception as e:
                mapping[uid] = None
                log(f"[FUTURE ERROR] {uid}: {e}")
            done_count += 1
            if done_count % 50 == 0 or done_count == len(unique_ids):
                log(f"Completed {done_count}/{len(unique_ids)} lookups")

log(f"Resolved {sum(v is not None for v in mapping.values())} / {len(unique_ids)} AUUIDs")

# Map back to DataFrame
df["Staff Name"] = df["AUUID_digits"].astype(str).map(mapping)

# -------------------- Save Output -------------------------------
out_cols = [HOSTNAME_COL, "AUUID_digits", "Staff Name"]
# Always write the file so you can inspect partial results
# If OUTPUT_FILE exists, append new rows; else, write new file

if os.path.exists(OUTPUT_FILE):
    # Read existing file
    existing_df = pd.read_excel(OUTPUT_FILE, sheet_name=OUTPUT_SHEET, engine="openpyxl")
    # Concatenate and drop duplicates based on 'Hostname' and 'AUUID_digits'
    combined_df = pd.concat([existing_df, df[out_cols]], ignore_index=True)
    combined_df = combined_df.drop_duplicates(subset=["Hostname", "AUUID_digits"])
    combined_df.to_excel(OUTPUT_FILE, index=False, sheet_name=OUTPUT_SHEET)
    log(f"Appended and wrote {len(combined_df)} rows to {OUTPUT_FILE} ({OUTPUT_SHEET}).")
else:
    df[out_cols].to_excel(OUTPUT_FILE, index=False, sheet_name=OUTPUT_SHEET)
    log(f"Done. Wrote {len(df)} rows to {OUTPUT_FILE} ({OUTPUT_SHEET}).")


Total rows: 701 | Unique AUUIDs to query: 624
Starting parallel lookups with 32 workers...
Completed 50/624 lookups


In [ ]:
df = pd.read_excel(RAW_FILE, sheet_name=SHEET_NAME, engine = "openpyxl")
df = df.assign(**{HOSTNAME_COL: df[HOSTNAME_COL].astype(str).str.split(r'[\n\r]+')})
df = df.explode(HOSTNAME_COL).reset_index(drop=True)
df["AUUID_digits"] = df["Hostname"].astype(str).str.extract(r"(\d{5,})")[0].astype(str)

def extract_staff_name(auuid):
    command = f'net user /domain "{auuid}"'
    result = subprocess.run(command, capture_output=True, text=True, shell=True)
    for line in result.stdout.splitlines():
        if 'Full Name' in line:
            parts = line.split(':', 1)
            if len(parts) > 1:
                return parts[1].strip()
    return None

df["Staff Name"] = df["AUUID_digits"].apply(extract_staff_name)
df[[HOSTNAME_COL, "AUUID_digits", "Staff Name"]].to_excel("output.xlsx", index=False, sheet_name='user data')
